In [5]:
"""
lista_completa_asselin_difusao.py

Script UNICO e AUTOSSUFICIENTE para toda a Lista de Exercicios (Questoes 1-5).
Nao depende de nenhum outro arquivo -- so precisa de numpy, matplotlib e scipy
instalados. Basta rodar (python lista_completa_asselin_difusao.py) ou colar
tudo em uma celula do Jupyter/Colab.

Cada questao roda automaticamente e IMPRIME a tabela pronta para copiar na
lista. As imagens pedidas sao salvas como arquivos .png na mesma pasta.

O numero de aneis (usado nas Questoes 2, 3 e 4) e contado por CODIGO (nao
"a olho"), usando scipy.signal.find_peaks sobre um corte radial a partir do
centro -- isso garante que todos os grupos cheguem exatamente ao mesmo
numero, eliminando a variabilidade de contagem visual.
"""

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LightSource, LinearSegmentedColormap
from scipy.signal import find_peaks


# =============================================================================
# ---- MODELO -----------------------------------------------------------------
# =============================================================================
class Fields2D:
    def __init__(self, Nx=150, Ny=150, dx=1.0, dy=1.0, dt=0.01,
                 Grav=9.8, H0=1.0, alfa=0.95, beta=0.99,
                 Kdif=1.0, diff_scale=0.03, use_4th_order=True):
        self.Nx, self.Ny = Nx, Ny
        self.dx, self.dy, self.dt = dx, dy, dt
        self.Grav, self.H0 = Grav, H0
        self.alfa, self.beta = alfa, beta
        self.Kdif = Kdif
        self.diff_scale = diff_scale
        self.use_4th_order = use_4th_order
        self.C = np.sqrt(Grav * H0)

        self.x = np.arange(Nx) * dx
        self.y = np.arange(Ny) * dy
        self.X, self.Y = np.meshgrid(self.x, self.y)

        self.u = np.zeros((Ny, Nx))
        self.v = np.zeros((Ny, Nx))
        self.h = np.zeros((Ny, Nx))
        self.um = np.zeros((Ny, Nx))
        self.vm = np.zeros((Ny, Nx))
        self.hm = np.zeros((Ny, Nx))

    def init_droplet(self, amp=0.9, sigma=22.5, wavelength=15.0):
        xc, yc = self.x[-1] / 2.0, self.y[-1] / 2.0
        r = np.sqrt((self.X - xc)**2 + (self.Y - yc)**2)
        k0 = 2.0 * np.pi / wavelength
        h0 = -amp * np.exp(-(r**2) / (2.0 * sigma**2)) * np.cos(k0 * r)
        self.h[:] = h0
        self.hm[:] = h0
        self.u[:] = 0.0
        self.v[:] = 0.0
        self.um[:] = 0.0
        self.vm[:] = 0.0


def ddx(Q, dx):
    return (np.roll(Q, -1, axis=1) - np.roll(Q, 1, axis=1)) / (2.0 * dx)


def ddy(Q, dy):
    return (np.roll(Q, -1, axis=0) - np.roll(Q, 1, axis=0)) / (2.0 * dy)


def laplacian(Q, dx, dy):
    d2x = (np.roll(Q, -1, axis=1) - 2.0 * Q + np.roll(Q, 1, axis=1)) / dx**2
    d2y = (np.roll(Q, -1, axis=0) - 2.0 * Q + np.roll(Q, 1, axis=0)) / dy**2
    return d2x + d2y


def computational_diffusion_2d(Q, dt, dx, dy, C, use_4th_order=True, scale=1.0):
    mu = C * dt / dx
    Kc = scale * (C * dx / 2.0) * (1.0 - mu)
    lap = laplacian(Q, dx, dy)
    diffterm = Kc * lap
    if use_4th_order:
        eps = Kc * dx**2 / 12.0
        diffterm = diffterm - eps * laplacian(lap, dx, dy)
    return diffterm


def rhs_u(h, Grav, dx):
    return -Grav * ddx(h, dx)


def rhs_v(h, Grav, dy):
    return -Grav * ddy(h, dy)


def rhs_h(u, v, H0, dx, dy):
    return -H0 * (ddx(u, dx) + ddy(v, dy))


def filter_raw(Qnew, Qc, Qm, alfa, beta):
    deslc = alfa * (Qm - 2.0 * Qc + Qnew)
    Qc_filt = Qc + deslc
    Qnew_filt = Qnew + deslc * (beta - 1.0)
    return Qc_filt, Qnew_filt


def step(f):
    """RK4 (4 estagios) + filtro RAW + difusao 2a/4a ordem."""
    dt, dx, dy = f.dt, f.dx, f.dy
    u, v, h = f.u, f.v, f.h
    um, vm, hm = f.um, f.vm, f.hm

    k1u, k1v, k1h = rhs_u(h, f.Grav, dx), rhs_v(h, f.Grav, dy), rhs_h(u, v, f.H0, dx, dy)
    u2, v2, h2 = u + 0.5*dt*k1u, v + 0.5*dt*k1v, h + 0.5*dt*k1h
    k2u, k2v, k2h = rhs_u(h2, f.Grav, dx), rhs_v(h2, f.Grav, dy), rhs_h(u2, v2, f.H0, dx, dy)
    u3, v3, h3 = u + 0.5*dt*k2u, v + 0.5*dt*k2v, h + 0.5*dt*k2h
    k3u, k3v, k3h = rhs_u(h3, f.Grav, dx), rhs_v(h3, f.Grav, dy), rhs_h(u3, v3, f.H0, dx, dy)
    u4, v4, h4 = u + dt*k3u, v + dt*k3v, h + dt*k3h
    k4u, k4v, k4h = rhs_u(h4, f.Grav, dx), rhs_v(h4, f.Grav, dy), rhs_h(u4, v4, f.H0, dx, dy)

    Unp1 = u + (dt/6.0) * (k1u + 2*k2u + 2*k3u + k4u)
    Vnp1 = v + (dt/6.0) * (k1v + 2*k2v + 2*k3v + k4v)
    Hnp1 = h + (dt/6.0) * (k1h + 2*k2h + 2*k3h + k4h)

    u_filt, Unp1 = filter_raw(Unp1, u, um, f.alfa, f.beta)
    v_filt, Vnp1 = filter_raw(Vnp1, v, vm, f.alfa, f.beta)
    h_filt, Hnp1 = filter_raw(Hnp1, h, hm, f.alfa, f.beta)

    if f.Kdif > 0:
        Unp1 = Unp1 + dt * computational_diffusion_2d(u_filt, dt, dx, dy, f.C, f.use_4th_order, f.diff_scale)
        Vnp1 = Vnp1 + dt * computational_diffusion_2d(v_filt, dt, dx, dy, f.C, f.use_4th_order, f.diff_scale)
        Hnp1 = Hnp1 + dt * computational_diffusion_2d(h_filt, dt, dx, dy, f.C, f.use_4th_order, f.diff_scale)

    f.um, f.u = u_filt, Unp1
    f.vm, f.v = v_filt, Vnp1
    f.hm, f.h = h_filt, Hnp1


def step_leapfrog(f):
    """Leapfrog/CTCS (centrado no tempo, centrado no espaco)."""
    dt, dx, dy = f.dt, f.dx, f.dy
    u, v, h = f.u, f.v, f.h
    um, vm, hm = f.um, f.vm, f.hm

    Unp1 = um + 2.0 * dt * rhs_u(h, f.Grav, dx)
    Vnp1 = vm + 2.0 * dt * rhs_v(h, f.Grav, dy)
    Hnp1 = hm + 2.0 * dt * rhs_h(u, v, f.H0, dx, dy)

    u_filt, Unp1 = filter_raw(Unp1, u, um, f.alfa, f.beta)
    v_filt, Vnp1 = filter_raw(Vnp1, v, vm, f.alfa, f.beta)
    h_filt, Hnp1 = filter_raw(Hnp1, h, hm, f.alfa, f.beta)

    if f.Kdif > 0:
        Unp1 = Unp1 + dt * computational_diffusion_2d(u_filt, dt, dx, dy, f.C, f.use_4th_order, f.diff_scale)
        Vnp1 = Vnp1 + dt * computational_diffusion_2d(v_filt, dt, dx, dy, f.C, f.use_4th_order, f.diff_scale)
        Hnp1 = Hnp1 + dt * computational_diffusion_2d(h_filt, dt, dx, dy, f.C, f.use_4th_order, f.diff_scale)

    f.um, f.u = u_filt, Unp1
    f.vm, f.v = v_filt, Vnp1
    f.hm, f.h = h_filt, Hnp1


def run_simulation(tend, scheme='RK4', **field_kwargs):
    f = Fields2D(**field_kwargs)
    f.init_droplet()
    step_fn = step if scheme == 'RK4' else step_leapfrog
    nsteps = int(round(tend / f.dt))
    for _ in range(nsteps):
        step_fn(f)
    return f


WATER_CMAP = LinearSegmentedColormap.from_list(
    'water', ['#0a3d52', '#0e6e8c', '#3fb8d6', '#8fe0ef', '#d9f7fb'])


def plot_surface(H2D, ax=None, vert_exag=25, title=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(6, 6))
    ls = LightSource(azdeg=315, altdeg=55)
    rgb = ls.shade(H2D, cmap=WATER_CMAP, vert_exag=vert_exag,
                    blend_mode='soft', vmin=H2D.min(), vmax=max(H2D.max(), 1e-6))
    ax.imshow(rgb, origin='lower')
    ax.set_xticks([]); ax.set_yticks([])
    ax.set_aspect('equal')
    if title:
        ax.set_title(title, fontsize=11)
    return ax


def metrics(f):
    """max|h| final e numero de aneis, contados por codigo (find_peaks
    sobre um corte radial a partir do centro) -- NAO conte a olho."""
    row = np.abs(f.h[f.Ny // 2, f.Nx // 2:])
    max_h = float(np.max(np.abs(f.h)))
    prom = 0.05 * max_h if max_h > 0 else 0
    peaks, _ = find_peaks(row, prominence=prom)
    return max_h, int(len(peaks))


def spectral_ratio(f):
    row = f.h[f.Ny // 2, :]
    F = np.abs(np.fft.rfft(row))**2
    n = len(F)
    low = F[1:max(2, n // 20)].sum()
    high = F[n // 3:].sum()
    return float(high / (low + 1e-30))


PARAMS_PADRAO = dict(Nx=150, Ny=150, dx=1.0, dy=1.0, dt=0.01, Kdif=1.0)
TEND_PADRAO = 25.0


# =============================================================================
# ---- QUESTAO 1: filtro de Asselin + modo computacional (Leapfrog) ---------
# =============================================================================
def questao1():
    print("\n" + "=" * 70)
    print("QUESTAO 1 -- Leapfrog + modo computacional semeado")
    print("=" * 70)
    print(f"{'alfa':>6} | {'D_10':>10} | {'D_2500':>10} | {'D2500/D10':>10} | {'r=1-2a':>8}")
    for alfa in [0.0, 0.5, 0.95]:
        f = Fields2D(Nx=150, Ny=150, dx=1.0, dy=1.0, dt=0.01, alfa=alfa, Kdif=0.0)
        f.init_droplet()
        f.hm = f.h + 0.05*np.exp(-((f.X-f.x[-1]/2)**2+(f.Y-f.y[-1]/2)**2)/(2*30.0**2))
        probe = (f.Ny//2, f.Nx//2 + 20)
        vals = []
        for it in range(2500):
            step_leapfrog(f)
            vals.append(f.h[probe])
        vals = np.array(vals)
        D_10 = abs(vals[10] - vals[9])
        D_2500 = abs(vals[-1] - vals[-2])
        r = 1 - 2*alfa
        print(f"{alfa:6.2f} | {D_10:10.6f} | {D_2500:10.6f} | {D_2500/D_10:10.4f} | {r:8.3f}")


# =============================================================================
# ---- QUESTAO 2: RAW (beta<1) vs RA puro (beta=1) --------------------------
# =============================================================================
def questao2():
    print("\n" + "=" * 70)
    print("QUESTAO 2 -- RAW (beta=0,99) vs RA puro (beta=1,00)")
    print("=" * 70)
    print(f"{'beta':>6} | {'max|h| final':>14} | {'n aneis':>8}")
    fig, axs = plt.subplots(1, 2, figsize=(11, 5.5))
    for ax, beta in zip(axs, [1.0, 0.99]):
        f = run_simulation(TEND_PADRAO, scheme='RK4', alfa=0.95, beta=beta,
                            diff_scale=0.03, use_4th_order=True, **PARAMS_PADRAO)
        max_h, n_rings = metrics(f)
        print(f"{beta:6.2f} | {max_h:14.5f} | {n_rings:8d}")
        plot_surface(f.h, ax=ax, title=f"beta={beta}")
    plt.tight_layout()
    plt.savefig("questao2_raw_vs_ra.png", dpi=130, bbox_inches="tight", facecolor="white")
    print("Figura salva: questao2_raw_vs_ra.png")


# =============================================================================
# ---- QUESTAO 3: difusao de 2a ordem (Kc) ----------------------------------
# =============================================================================
def questao3():
    print("\n" + "=" * 70)
    print("QUESTAO 3 -- Difusao computacional de 2a ordem (Kc)")
    print("=" * 70)
    print(f"{'diff_scale':>10} | {'max|h| final':>14} | {'n aneis':>8}")
    fig, axs = plt.subplots(1, 5, figsize=(22, 5))
    diff_scales = [0.0, 0.01, 0.03, 0.10, 0.50]
    for ax, ds in zip(axs, diff_scales):
        f = run_simulation(TEND_PADRAO, scheme='RK4', alfa=0.95, beta=0.99,
                            diff_scale=ds, use_4th_order=False, **PARAMS_PADRAO)
        max_h, n_rings = metrics(f)
        print(f"{ds:10.2f} | {max_h:14.5f} | {n_rings:8d}")
        plot_surface(f.h, ax=ax, title=f"diff_scale={ds}")
    plt.tight_layout()
    plt.savefig("questao3_difusao_2a_ordem.png", dpi=130, bbox_inches="tight", facecolor="white")
    print("Figura salva: questao3_difusao_2a_ordem.png")

    C = np.sqrt(9.8 * 1.0)
    dx, dt = 1.0, 0.01
    mu = C * dt / dx
    Kc = (C * dx / 2.0) * (1.0 - mu)
    print(f"\n(c) C={C:.4f}  mu={mu:.6f}  Kc(scale=1,0) = {Kc:.4f}")


# =============================================================================
# ---- QUESTAO 4: difusao de 4a ordem (hiperdifusao) -------------------------
# =============================================================================
def questao4():
    print("\n" + "=" * 70)
    print("QUESTAO 4 -- Difusao computacional de 4a ordem (hiperdifusao)")
    print("=" * 70)
    print(f"{'use_4th_order':>14} | {'max|h| final':>14} | {'n aneis':>8} | {'high/low':>12}")
    fig, axs = plt.subplots(1, 2, figsize=(11, 5.5))
    for ax, use4 in zip(axs, [False, True]):
        f = run_simulation(TEND_PADRAO, scheme='RK4', alfa=0.95, beta=0.99,
                            diff_scale=0.03, use_4th_order=use4, **PARAMS_PADRAO)
        max_h, n_rings = metrics(f)
        ratio = spectral_ratio(f)
        print(f"{str(use4):>14} | {max_h:14.5f} | {n_rings:8d} | {ratio:12.6g}")
        plot_surface(f.h, ax=ax, title=f"use_4th_order={use4}")
    plt.tight_layout()
    plt.savefig("questao4_difusao_4a_ordem.png", dpi=130, bbox_inches="tight", facecolor="white")
    print("Figura salva: questao4_difusao_4a_ordem.png")


# =============================================================================
# ---- QUESTAO 5: sintese (usa as tabelas das Questoes 2-4, nao roda nada novo)
# =============================================================================
def questao5():
    print("\n" + "=" * 70)
    print("QUESTAO 5 -- Sintese")
    print("=" * 70)
    print("Use os numeros ja impressos nas Questoes 2, 3 e 4 acima para")
    print("escolher beta, diff_scale e use_4th_order. Nao e necessario")
    print("rodar nenhuma simulacao nova para esta questao.")


if __name__ == '__main__':
    #questao1()
    #questao2()
    #questao3()
    #questao4()
    #questao5()
    print("\nTODAS AS QUESTOES CONCLUIDAS.")


TODAS AS QUESTOES CONCLUIDAS.
